# Analysis: Step 0

Here is a small utility script which speeds up the process of fetching the papers for our analyses.

This script will open a browser to allow the user to download the paper, put them in the designated folder, and open the next tab once the paper is downloaded.

In [ ]:
# Path to a CSV containing all of the papers, exported from Covidence.
#   This should contain a `DOI` column.
INCLUDED_PAPERS_CSV: str = "./included.csv"

# Path to the directory where papers will be downloaded to.
DOWNLOAD_DIR: str = "papers/"

In [ ]:
import pandas as pd
import requests
from urllib.parse import urlsplit

df = pd.read_csv(INCLUDED_PAPERS_CSV)
raw_dois = df["DOI"]

# Some DOI fields were already links, so make the rest proper links.
is_link = raw_dois.str.startswith("https://")
links = pd.concat([raw_dois[is_link], ("https://doi.org/" + raw_dois[~is_link])]).sort_index()

redirects_to: list[str] = []

for link in links:
    res = requests.head(link)
    redirects_to.append(res.headers.get("location"))

_, df["host"], df["path"], *_ = zip(*(tuple(urlsplit(canonical)) for canonical in redirects_to))
df["canonical"] = redirects_to

In [ ]:
import subprocess
import typing
import webbrowser
from watchdog.observers import Observer
from watchdog.observers.api import BaseObserver
from watchdog.events import FileSystemEventHandler, FileCreatedEvent, DirCreatedEvent

class WatchForDownloads(FileSystemEventHandler):
    def __init__(self, observer: BaseObserver) -> None:
        super().__init__()
        self.observer = observer
        self.observer.start()
        self.waiting = True

    @typing.override
    def on_created(self, event: FileCreatedEvent | DirCreatedEvent):
        self.waiting = False


observer = Observer()
handler = WatchForDownloads(observer)
observer.schedule(handler, DOWNLOAD_DIR, recursive=False)

for i, url in enumerate(df["canonical"]):
    # Copies id to the clipboard.
    # This is a little hack because a lot of Python clipboard libraries do not support Wayland.
    subprocess.call(["wl-copy", f"{i}"])
    webbrowser.open(url)

    while handler.waiting:
        continue

    handler.waiting = True

observer.stop()

In [8]:
df.to_csv("exported.csv", index=True, columns=["Authors", "Published Year", "Title", "DOI"])